# Regression as a thinking tool

Core, Part 2 (Statistics II and decisions), lesson C11. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** read the slope and intercept of a fitted line in the units of the data, compute residuals and say what they show, interpret R-squared without overclaiming, add a second variable and read a coefficient as "holding the other fixed", and explain why extrapolating is risky.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each line: its amount in dollars (quantity × price − discount) and the units that are food or bakery.
price = dict(zip(products["product_id"], products["unit_price_usd"]))
category = dict(zip(products["product_id"], products["category"]))
lines = order_items.assign(
    amount=order_items["quantity"] * order_items["product_id"].map(price) - order_items["discount_usd"],
    food_units=order_items["quantity"].where(order_items["product_id"].map(category) == "food", 0),
    bakery_units=order_items["quantity"].where(order_items["product_id"].map(category) == "bakery", 0),
)
# One row for each completed order: its total in dollars and its units (the sum of the quantities of its lines).
per_order = lines.groupby("order_id").agg(total=("amount", "sum"), units=("quantity", "sum"), food_units=("food_units", "sum"), bakery_units=("bakery_units", "sum"))
sales = orders[orders["status"] == "completed"].merge(per_order, on="order_id", validate="one_to_one")
assert len(sales) == 1799  # the canceled order of 5.00 is left out
print("1,799 completed orders. `sales` has total, units, food_units and bakery_units for each one.")

## The question

In [ ]:
print(sales[["order_id", "units", "total"]].head(4).to_string(index=False))

## Unit 1: A line in the units of the data

**Predict first.** Fit the line of the order total on the number of units, over all 1,799 orders. How many dollars will each extra unit add (the slope), and what total will the line give at 0 units (the intercept)?

In [ ]:
units, total = sales["units"], sales["total"]
slope, intercept = np.polyfit(units, total, 1)
print(round(slope, 2), round(intercept, 2))
print(round(intercept + slope * 3, 2), round(total[units == 3].mean(), 2), units.min())
sxy = ((units - units.mean()) * (total - total.mean())).sum()
sxx = ((units - units.mean()) ** 2).sum()
print(round(sxy, 2), round(sxx, 2), round(units.mean(), 4), round(total.mean(), 4), round(sxy / sxx, 4))

## Unit 2: Residuals

**Predict first.** What are the smallest and the largest residual of this line, and what do all 1,799 residuals add up to?

In [ ]:
residual = total - (intercept + slope * units)
worst = sales.assign(residual=residual).sort_values("residual")
print(round(residual.min(), 2), round(residual.max(), 2), abs(round(residual.sum(), 6)))
top = worst.tail(1)
print(top[["order_id", "store_id", "units", "total", "residual"]].round(2).to_string(index=False))
print(round(float(intercept + slope * top["units"].iloc[0]), 2))

**Predict first.** Leave out the order with the largest residual and fit again. Will the slope move by cents or by dollars, and will the intercept get closer to 0? And will the misses of the new line get larger or smaller for bigger orders?

In [ ]:
clean = sales[sales["order_id"] != 2075]  # from here on: the 1,798 orders without order 2075
slope2, intercept2 = np.polyfit(clean["units"], clean["total"], 1)
print(len(clean), clean["units"].max(), round(slope2, 2), round(intercept2, 2))
miss = clean["total"] - (intercept2 + slope2 * clean["units"])
print(miss.groupby(clean["units"]).std().loc[1:8].round(2).to_dict())

## Unit 3: R-squared

**Predict first.** R² compares the squared misses of the line with the squared distances from the mean total. Will it be close to 0.2, 0.5 or 0.8 for the 1,798 orders?

In [ ]:
ss_res = (miss ** 2).sum()
ss_tot = ((clean["total"] - clean["total"].mean()) ** 2).sum()
print(round(ss_res, 1), round(ss_tot, 1), round(1 - ss_res / ss_tot, 3))

**Predict first.** Now fit the line only to orders of 1 or 2 units. Will the slope change a lot, and will R² stay the same?

In [ ]:
small = clean[clean["units"] <= 2]
slope3, intercept3 = np.polyfit(small["units"], small["total"], 1)
r2_small = np.corrcoef(small["units"], small["total"])[0, 1] ** 2
print(len(small), round(slope3, 2), round(r2_small, 3))

## Unit 4: Holding the other fixed

**Predict first.** Add the units of food in the order as a second variable. Will the coefficient on units stay near 4.41 or move? And how many dollars will one more food unit add, holding the number of units fixed?

In [ ]:
X = np.column_stack([np.ones(len(clean)), clean["units"], clean["food_units"]])
coef = np.linalg.lstsq(X, clean["total"], rcond=None)[0]
print([round(float(c), 2) for c in coef])
miss2 = clean["total"] - X @ coef
print(round(1 - (miss2 ** 2).sum() / ss_tot, 3))
print(round(float(coef @ [1, 3, 0]), 2), round(float(coef @ [1, 3, 1]), 2))

**Predict first.** Check the two coefficients against the menu: what does the average unit cost after discounts, for units that are not food and for units that are food?

In [ ]:
paid = lines[lines["order_id"] != 2075]
is_food = paid["food_units"] > 0
print(round(paid.loc[~is_food, "amount"].sum() / paid.loc[~is_food, "quantity"].sum(), 2), round(paid.loc[is_food, "amount"].sum() / paid.loc[is_food, "quantity"].sum(), 2))

## Unit 5: Extrapolation

**Predict first.** Count the completed orders so far, day by day, and fit a line to January 1 to March 31 (days 1 to 90). Will R² be close to 0.2, 0.5 or 1? Then carry the line on to June 30 (day 181): will it land near the real count of 1,799?

In [ ]:
daily = sales.groupby(sales["ordered_at"].str[:10]).size().reindex(pd.date_range("2026-01-01", "2026-06-30").strftime("%Y-%m-%d"), fill_value=0)
running = daily.cumsum().to_numpy()
day = np.arange(1, len(daily) + 1)
q1 = day <= 90  # January 1 to March 31
b_q1, a_q1 = np.polyfit(day[q1], running[q1], 1)
r2_q1 = np.corrcoef(day[q1], running[q1])[0, 1] ** 2
print(round(b_q1, 2), round(a_q1, 2), round(r2_q1, 4))
print(round(a_q1 + b_q1 * 181), int(running[-1]))
print(round(daily[:90].mean(), 2), round(daily[90:].mean(), 2))

## Finish it

### Exercise 1: Finish it

**Task:** How many dollars does each extra unit add to the total of a Campus order, store 2? Fit the line of `total` on `units` to the Campus orders and put its slope, rounded to 2 decimals, in `answer_1`.

**Expected:** `answer_1` is a number in dollars per unit.

In [ ]:
campus = clean[clean["store_id"] == 2]  # given: the Campus orders
# fit the line of total on units for campus; set answer_1 to its slope, rounded to 2 decimals
answer_1 = None  # replace None

In [ ]:
_ = check("Exercise 1", "answer_1", 4.52, traps={4.41: "That is the slope for the orders of all three stores. Fit only the Campus orders.", -0.29: "That is the intercept. `np.polyfit(x, y, 1)` returns the slope first."})

<details>
<summary>Hint</summary>

`np.polyfit(x, y, 1)` returns two numbers: the slope, then the intercept.

</details>

<details>
<summary>Solution</summary>

```python

campus = clean[clean["store_id"] == 2]
slope_c, intercept_c = np.polyfit(campus["units"], campus["total"], 1)
answer_1 = round(float(slope_c), 2)

```

4.52 dollars per unit at Campus, a little above the 4.41 of all stores together, so the stores do not add the same dollars per unit. The check cell reports the slope of all stores, or the intercept of Campus, if you give one of those.

</details>

## On your own

### Exercise 2: On your own

**Task:** What does one more bakery unit do to an order's total, holding the number of units fixed? Before you start, say in one sentence what the result looks like: one number in dollars. Fit the total on two variables, `units` and `bakery_units`, over the orders of `clean`, and put the coefficient of `bakery_units`, rounded to 2 decimals, in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a number in dollars.

In [ ]:
_ = check("Exercise 2", "answer_2", -1.28, traps={1.16: "That is the slope of the total on bakery units alone. The task holds the number of units fixed: put both variables in the fit."})

<details>
<summary>Hint</summary>

Build the table of the fit like the one of unit 4: a column of ones, then `units`, then `bakery_units`. The coefficients come back in that order.

</details>

<details>
<summary>Solution</summary>

```python

X = np.column_stack([np.ones(len(clean)), clean["units"], clean["bakery_units"]])
coef = np.linalg.lstsq(X, clean["total"], rcond=None)[0]
answer_2 = round(float(coef[2]), 2)

```

−1.28 dollars: among orders with the same number of units, one bakery unit instead of another kind of unit lowers the total by 1.28, because a bakery unit costs less than the average of the other units. The slope of the total on bakery units alone is +1.16, because orders with more bakery units also have more units: the sign flips once the units are held fixed, like the reversal in *Correlation is not causation* (C10). The check cell reports 1.16.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants the dollars that each extra unit adds at Harbor, store 3. The colleague fits the line the other way round, `units` on `total`, and divides 1 by its slope. Fix the code so that it fits `total` on `units`, and set `answer_3` to the slope, rounded to 2 decimals.

**Expected:** `answer_3` is a number in dollars per unit.

In [ ]:
harbor = clean[clean["store_id"] == 3]  # the Harbor orders
slope_back, intercept_back = np.polyfit(harbor["total"], harbor["units"], 1)  # the colleague fits units on total
answer_3 = round(float(1 / slope_back), 2)  # and turns the slope upside down

In [ ]:
_ = check("Exercise 3", "answer_3", 4.37, traps={5.69: "That is 1 divided by the slope of units on total. The line of total on units is not the same line turned around."})

<details>
<summary>Hint</summary>

A fitted line predicts the column on its left side from the column on its right side. Which column should be predicted here?

</details>

<details>
<summary>Solution</summary>

```python

harbor = clean[clean["store_id"] == 3]
slope_h, intercept_h = np.polyfit(harbor["units"], harbor["total"], 1)
answer_3 = round(float(slope_h), 2)

```

4.37 dollars per unit at Harbor. The colleague's number is 5.69: the line of units on total is a different line, because it minimizes misses in units, not in dollars, and turning its slope upside down overshoots whenever the points do not lie exactly on a line. The check cell reports 5.69.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")